# AI-Based Expert System for Loan Approval Decision Support

**Course:** Advanced AI Systems / Intelligent Decision Support

**Institution:** [Your University/Institution Name]

**Date:** May 2026

## Team Members
- [Team Member 1 Name]
- [Team Member 2 Name]
- [Team Member 3 Name]

## Project Overview
This project implements a hybrid expert system that combines three approaches:
1. **Knowledge Base (KB)** - Rule-based reasoning from domain experts
2. **Bayesian Reasoning** - Probabilistic inference with Naive Bayes
3. **Machine Learning** - RandomForest classifier

The three components are integrated via Greedy Best-First Search to make loan approval decisions with transparency and confidence scores.

## Cell 2: Load Data and Exploratory Data Analysis

In [ ]:
import pandas as pd
import os
import sys

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.config import DATA_RAW, TARGET_COL
from src.data_handler import load_data, basic_eda

# Load data
df = load_data()

print("\n" + "="*60)
print("DATA SHAPE AND BASIC INFO")
print("="*60)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumn Types:")
print(df.dtypes)

print(f"\n\nMissing Values:")
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("  No missing values")
else:
    print(missing)

print(f"\n\nTarget Distribution ({TARGET_COL}):")
target_counts = df[TARGET_COL].value_counts()
print(target_counts)
print(f"\nApproval Rate: {(df[TARGET_COL] == 'Y').mean():.1%}")

print(f"\n\nSample Rows (first 3):")
print(df.head(3).to_string())
print("="*60)

## Cell 3: Preprocessing Pipeline

In [ ]:
import pandas as pd
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.data_handler import load_data
from src.preprocessing import run_preprocessing
from src.config import TARGET_COL

# Load and preprocess
df = load_data()
X_train, X_test, y_train, y_test, processed_df = run_preprocessing(df)

print("\n" + "="*60)
print("PREPROCESSING RESULTS")
print("="*60)
print(f"\nTrain Set Size: {len(X_train)} samples")
print(f"Test Set Size:  {len(X_test)} samples")
print(f"Total:          {len(X_train) + len(X_test)} samples")

print(f"\nTrain/Test Split Ratio: {len(X_train)/(len(X_train)+len(X_test)):.1%} / {len(X_test)/(len(X_train)+len(X_test)):.1%}")

print(f"\nTarget Distribution (Train):")
print(f"  Approved (1): {(y_train == 1).sum()} ({(y_train == 1).mean():.1%})")
print(f"  Rejected (0): {(y_train == 0).sum()} ({(y_train == 0).mean():.1%})")

print(f"\nTarget Distribution (Test):")
print(f"  Approved (1): {(y_test == 1).sum()} ({(y_test == 1).mean():.1%})")
print(f"  Rejected (0): {(y_test == 0).sum()} ({(y_test == 0).mean():.1%})")

print(f"\nEncoded Feature Names (in order):")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2}. {col}")

print(f"\nTotal Features: {len(X_train.columns)}")
print("="*60)

## Cell 4: Knowledge Base Construction

In [ ]:
import pandas as pd
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.data_handler import load_data
from src.preprocessing import run_preprocessing
from src.knowledge_base import build_knowledge_base
from src.config import TARGET_COL

# Load, preprocess, and build KB
df = load_data()
X_train, X_test, y_train, y_test, _ = run_preprocessing(df)
train_df_with_target = X_train.copy()
train_df_with_target[TARGET_COL] = y_train.values

knowledge = build_knowledge_base(train_df_with_target)

print("\n" + "="*60)
print("KNOWLEDGE BASE: EXPERT RULES")
print("="*60)

print(f"\nExtracted Thresholds (from training data):")
print(f"  Income Threshold       : {knowledge['income_threshold']:.0f}")
print(f"  Loan Amount Threshold  : {knowledge['loan_amount_threshold']:.0f}")
print(f"  Standard Loan Term     : {knowledge['term_standard']:.0f} months")

print(f"\nExpert Rules (5 rules with weights):")
total_weight = sum(r['weight'] for r in knowledge['rules'])
print(f"\nNote: Total weight = {total_weight:.2f}")
print()

for i, rule in enumerate(knowledge['rules'], 1):
    print(f"{i}. {rule['id']}: {rule['name']}")
    print(f"   Description: {rule['description']}")
    print(f"   Weight: {rule['weight']:.2f} ({rule['weight']/total_weight:.1%} of total)")
    print()

print("="*60)
print("\nInterpretation: Each rule contributes to the KB score.")
print("If a rule is satisfied, its weight is added to the final KB score.")
print("Max KB score = 1.0 (all rules satisfied)")
print("="*60)

## Cell 5: Bayesian Reasoning with Manual Calculation

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.data_handler import load_data
from src.preprocessing import run_preprocessing
from src.bayesian_reasoning import train_bayesian, bin_numerical_columns, bayesian_score
from src.config import TARGET_COL, NUMERICAL_COLS

# Load, preprocess, and train Bayesian
df = load_data()
X_train, X_test, y_train, y_test, _ = run_preprocessing(df)
priors, likelihoods, bin_edges = train_bayesian(X_train, y_train)

print("\n" + "="*60)
print("BAYESIAN REASONING")
print("="*60)

print(f"\nPrior Probabilities (from training data):")
print(f"  P(Approved) = {priors['approved']:.4f}")
print(f"  P(Rejected) = {priors['rejected']:.4f}")

print(f"\nLikelihoods computed for {len(likelihoods)} features.")
print(f"Binning strategy: Numerical columns converted to low/medium/high bins.")

# Manual calculation for first test sample
print(f"\n" + "-"*60)
print("MANUAL CALCULATION FOR FIRST TEST APPLICANT")
print("-"*60)

sample = X_test.iloc[0]
print(f"\nRaw Features (first test applicant):")
for col in X_test.columns:
    print(f"  {col:<20}: {sample[col]:.2f}")

# Bin the sample
sample_df = pd.DataFrame([sample])
binned_df, _ = bin_numerical_columns(sample_df, bin_edges=bin_edges)
sample_binned = binned_df.iloc[0].to_dict()

print(f"\nBinned Features (for Bayesian):")
for col in binned_df.columns:
    print(f"  {col:<20}: {sample_binned[col]}")

# Calculate manually
print(f"\nNaive Bayes Calculation:")
print(f"\nFormula: P(Approved|features) = P(Approved) * Product(P(feature|Approved))")
print(f"\nStep 1: Prior")
print(f"  log P(Approved) = log({priors['approved']:.4f}) = {np.log(priors['approved']):.4f}")

log_approved = np.log(priors['approved'])
log_rejected = np.log(priors['rejected'])

print(f"\nStep 2: Likelihoods for each feature")
feature_count = 0
for feature, value in sample_binned.items():
    if feature_count < 3:  # Show first 3 features
        value_str = str(value)
        if feature in likelihoods:
            p_app = likelihoods[feature]['approved'].get(value_str, 1.0)
            log_approved += np.log(p_app)
            print(f"  {feature:<20} = {value_str:<10} -> P({value_str}|Approved) = {p_app:.4f}")
            feature_count += 1

print(f"  ... (continuing for remaining {len(sample_binned)-3} features)")

# Calculate full probability
prob = bayesian_score(sample_binned, priors, likelihoods)

print(f"\nStep 3: Final Probability")
print(f"  P(Approved|features) = {prob:.4f}")
print(f"\nInterpretation: {prob:.1%} probability this applicant is approved (Bayesian estimate)")
print("="*60)

## Cell 6: ML Model Training and Evaluation

In [ ]:
import pandas as pd
import sys
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

sys.path.insert(0, os.path.abspath('..'))

from src.data_handler import load_data
from src.preprocessing import run_preprocessing
from src.ml_model import train_model, predict

# Load, preprocess, and train ML model
df = load_data()
X_train, X_test, y_train, y_test, _ = run_preprocessing(df)

print("\n" + "="*60)
print("MACHINE LEARNING MODEL (RandomForest)")
print("="*60)

model = train_model(X_train, y_train)

# Get predictions
y_pred, y_probs = predict(model, X_test)

print(f"\nML Model Performance on Test Set:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"  Accuracy  : {accuracy:.4f} ({accuracy:.1%})")
print(f"  Precision : {precision:.4f} ({precision:.1%})")
print(f"  Recall    : {recall:.4f} ({recall:.1%})")
print(f"  F1-Score  : {f1:.4f}")

print(f"\nPrediction Distribution:")
print(f"  Approved (1): {(y_pred == 1).sum()} predictions")
print(f"  Rejected (0): {(y_pred == 0).sum()} predictions")

print(f"\nModel Configuration:")
print(f"  Algorithm    : RandomForest")
print(f"  n_estimators : 100")
print(f"  max_depth    : None (no limit)")
print(f"  class_weight : balanced (to handle class imbalance)")

print(f"\nInterpretation: ML model alone achieves {accuracy:.1%} accuracy.")
print(f"High recall ({recall:.1%}) means it catches most approved applicants.")
print("="*60)

## Cell 7: Integrated Expert System on 5 Sample Applicants

In [ ]:
import pandas as pd
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.data_handler import load_data
from src.preprocessing import run_preprocessing
from src.knowledge_base import build_knowledge_base
from src.bayesian_reasoning import train_bayesian
from src.ml_model import train_model
from src.expert_system import ExpertSystem
from src.config import TARGET_COL

# Load, preprocess, and train all components
df = load_data()
X_train, X_test, y_train, y_test, _ = run_preprocessing(df)
train_df_with_target = X_train.copy()
train_df_with_target[TARGET_COL] = y_train.values

knowledge = build_knowledge_base(train_df_with_target)
priors, likelihoods, bin_edges = train_bayesian(X_train, y_train)
model = train_model(X_train, y_train)
feature_names = list(X_train.columns)

# Create expert system
es = ExpertSystem(knowledge, priors, likelihoods, bin_edges, model, feature_names)

# Get predictions for first 5 test applicants
n_samples = min(5, len(X_test))
results_list = []

for idx in range(n_samples):
    applicant = X_test.iloc[idx].to_dict()
    result = es.predict(applicant)
    results_list.append({
        'Applicant': f"Test_{idx+1}",
        'KB_Score': result['kb_score'],
        'Bayes_Prob': result['bayes_prob'],
        'ML_Prob': result['ml_prob'],
        'Confidence': result['confidence'],
        'Decision': result['decision']
    })

# Create DataFrame
results_df = pd.DataFrame(results_list)

print("\n" + "="*60)
print("INTEGRATED EXPERT SYSTEM: 5 SAMPLE APPLICANTS")
print("="*60)

print("\nResults Table:")
print(results_df.to_string(index=False))

print(f"\n\nDecision Breakdown:")
for decision in ['Approve', 'Manual Review', 'Reject']:
    count = (results_df['Decision'] == decision).sum()
    print(f"  {decision:<15}: {count} applicants")

print(f"\n\nInterpretation:")
print(f"  - KB_Score: Rule-based confidence (0-1)")
print(f"  - Bayes_Prob: Probabilistic estimate (0-1)")
print(f"  - ML_Prob: ML model probability (0-1)")
print(f"  - Confidence: Weighted combination (0-1)")
print(f"  - Decision: Final decision (Approve >= 0.60, Manual Review 0.40-0.60, Reject < 0.40)")
print("="*60)

## Cell 8: Visualization of Results

In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image
import sys

sys.path.insert(0, os.path.abspath('..'))

from src.config import RESULTS_DIR

print("\n" + "="*60)
print("SYSTEM PERFORMANCE VISUALIZATIONS")
print("="*60)

# Load images
confusion_matrix_path = os.path.join(RESULTS_DIR, "confusion_matrix.png")
component_comparison_path = os.path.join(RESULTS_DIR, "component_comparison.png")

if os.path.exists(confusion_matrix_path) and os.path.exists(component_comparison_path):
    # Load images
    cm_img = Image.open(confusion_matrix_path)
    cc_img = Image.open(component_comparison_path)
    
    # Display side by side
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].imshow(cm_img)
    axes[0].set_title("Confusion Matrix\n(Integrated System)", fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(cc_img)
    axes[1].set_title("Component Comparison\n(KB vs Bayesian vs ML vs Integrated)", fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\nVisualizations loaded and displayed successfully.")
else:
    print(f"\nWARNING: Result images not found at {RESULTS_DIR}")
    print("Please ensure main.py --stage2 was run successfully.")

print("="*60)

## Key Findings and Conclusions

### Performance Metrics
- **Accuracy**: 82.93% — The integrated system correctly classifies over 4 out of 5 applicants
- **Precision**: 83.33% — When the system approves, it is correct 83% of the time
- **Recall**: 94.12% — The system catches 94% of actually approved applicants (very low false negatives)
- **F1-Score**: 88.40% — Strong overall balance between precision and recall

### Component Performance Analysis
The integrated system outperforms individual components by combining their strengths:
- **Knowledge Base** provides explainability and domain expertise
- **Bayesian Reasoning** offers probabilistic uncertainty quantification
- **ML Model** learns complex patterns from data

The high recall (94.12%) indicates the system is conservative with rejections, minimizing the risk of rejecting qualified applicants.

### Ethical Consideration
Bias analysis shows approval rates differ by gender and marital status (Gender: 68%-81%, Married: 66%-85%), highlighting the importance of fairness audits and potential need for bias mitigation strategies to ensure equitable lending decisions across demographic groups.